# 03 — Tier 1: direct LLM baseline (Ollama + Gemma)

Gemma answers the benchmark questions with **no access to the database**. Its
figures have no source evidence, even when a value happens to be correct.

This notebook explores the baseline's responses. Its original scalar matching
and plots are **illustrative only**: they do not check complete lists or public
claims. Use notebook 06 or `tests/run_examples.py --full --publish` for the
common scoring rubric and the report's comparison of all three tiers.

The model runs locally with no hosted API fee. Hardware and electricity costs
are not estimated.

**Setup**

Install the project requirements and start Ollama, then download the exact model:

```bash
ollama pull gemma3:4b
```



## 1. Setup


In [1]:
import json, time
from pathlib import Path

import ollama
import pandas as pd
import yaml
import matplotlib.pyplot as plt

MODEL = 'gemma3:4b'        # check what you have with: ollama list
TOLERANCE = 0.01        # a number counts as correct within 1%

RESULTS = Path('../eval/results')
RESULTS.mkdir(parents=True, exist_ok=True)

questions = yaml.safe_load(Path('../eval/benchmark.yaml').read_text())['questions']
print(len(questions), 'questions |', sum(q['answerable'] for q in questions),
      'answerable,', sum(not q['answerable'] for q in questions), 'must refuse')


30 questions | 25 answerable, 5 must refuse


## 2. The prompt

We tell it what data exists but give it no way to query it. Asking for JSON keeps every
number it asserts as a separate claim, which is the same shape Tier 2 and Tier 3 use.


_Note: the prompt says there is no cost or margin data. If Gemma still answers the
profit-margin question, that is a strong result for the report._


In [ ]:
ANSWER_SHAPE = {
    "type": "object",
    "properties": {
        "findings": {"type": "string"},
        "claims": {"type": "array", "items": {
            "type": "object",
            "properties": {
                "text": {"type": "string"},
                "value": {"type": "number"},
                "kind": {"type": "string", "enum": ["number", "boolean"]},
                "row": {"type": "integer"},
                "column": {"type": "string"},
                "unit": {"type": "string"},
                "from_call": {"type": "integer"},
                "calc": {"type": "string",
                         "enum": ["none", "pct_change", "share", "sum", "diff",
                                  "difference", "ratio", "mean"]},
                "inputs": {"type": "array", "items": {"type": "number"}},
            },
            "required": ["text", "value", "unit", "from_call", "calc", "inputs"]}},
        "kpis": {"type": "object"},
        "limitations": {"type": "string"},
        "insufficient_data": {"type": "boolean"},
    },
    "required": ["findings", "claims"],
}

# The schema matches notebook 06 so both baseline runs measure the same setup.


## 3. Ask Gemma


In [8]:
def model_usage(responses):
    """Keep actual Ollama token counts; missing usage is unknown, not zero."""
    if not responses:
        return {'model_calls': 0, 'input_tokens': 0, 'output_tokens': 0}
    return {key: sum(r[key] for r in responses) if all(r.get(key) is not None for r in responses) else None
            for key in ('model_calls', 'input_tokens', 'output_tokens')}


def ask_gemma(system, question, shape):
    """Ask the model for JSON of a particular shape.

    `shape` is a JSON schema. Ollama forces the reply to match it, which is a
    lot more reliable than asking nicely and hoping. If the reply still will
    not parse we try once more with a blunter instruction.
    """
    import ollama

    usage = []
    for attempt in range(2):
        prompt = question if attempt == 0 else question + """

YOUR LAST REPLY WAS NOT VALID JSON.
Reply with ONE complete JSON object matching the schema. Keep the text short.
Do not write anything outside the JSON object."""

        try:
            reply = ollama.Client(timeout=120).chat(model=MODEL, format=shape,
                                options={"temperature": 0, "num_predict": 2048},
                                messages=[{"role": "system", "content": system},
                                          {"role": "user", "content": prompt}])
        except ollama.ResponseError as error:
            if error.status_code != 500:
                raise  # Missing models/authentication need an actionable setup error.
            usage.append({'model_calls': 1, 'input_tokens': None, 'output_tokens': None})
            # The caller can retry planning/SQL within its normal budget. An
            # optional chart failure must not discard a successful data query.
            return {'broken_json': True, 'error': 'Model generation failed: ' + str(error),
                    '_usage': model_usage(usage)}
        usage.append({'model_calls': 1, 'input_tokens': reply.get('prompt_eval_count'),
                      'output_tokens': reply.get('eval_count')})
        try:
            parsed = json.loads(reply["message"]["content"])
            if isinstance(parsed, dict):
                parsed['_usage'] = model_usage(usage)
                return parsed
        except Exception:
            continue

    return {"broken_json": True, "_usage": model_usage(usage)}


def tier1(question):
    """Same output contract, but no tools and no access to database results."""
    start = time.time()
    prompt = ("You are a business data analyst for a UK online gift wholesaler. "
              "Data covers December 2009 to December 2011. There is no cost, profit, "
              "marketing or demographic data. Answer the question with findings and "
              "numeric claims. Set insufficient_data if you cannot answer. You have no tools.")
    reply = ask_gemma(prompt, question, ANSWER_SHAPE)
    if reply.get("broken_json"):
        raise ValueError("Tier 1 did not return a usable JSON object")
    return {"question": question, "tier": 1, "findings": reply.get("findings", ""),
            "claims": reply.get("claims") or [], "kpis": {}, "fields_used": [],
            "filters_used": [], "chart": None, "plan": [],
            "route": "model", "scope_check": "numeric evidence only",
            "insufficient_data": bool(reply.get("insufficient_data")),
            "limitations": reply.get("limitations"), "log": [], "retries": 0, "usage": reply.get("_usage", {}),
            "seconds": round(time.time() - start, 1)}


def ask(question):
    """Keep the notebook's friendly column names while using the shared contract."""
    start = time.time()
    try:
        answer = tier1(question)
        values = [c.get('value') for c in answer['claims'] if isinstance(c, dict) and c.get('value') is not None]
        return {'answer': answer['findings'], 'values': values,
                'value': values[0] if values else None,
                'insufficient_data': answer['insufficient_data'],
                'latency_s': answer['seconds'], 'error': None}
    except Exception as error:
        return {'answer': 'The model call failed.', 'values': [], 'value': None,
                'insufficient_data': False, 'latency_s': round(time.time()-start, 1),
                'error': str(error)}


## 4. Try one first

Cheap check that Ollama is running before doing all 30.


In [ ]:
q = questions[0]
print('Q:   ', q['question'])
print('TRUE:', q['gt_answer'])
print('-' * 60)

a = ask(q['question'])
print('GEMMA:', a.get('answer'))
print('VALUE:', a.get('value'), a.get('unit', ''))
print('took ', a['latency_s'], 's')


## 5. Run all 30

A few minutes on a laptop. Slower than a hosted API, but free.


In [ ]:
rows = []
for i, q in enumerate(questions, 1):
    print(f"[{i:2}/{len(questions)}] {q['id']}", end=' ', flush=True)
    a = ask(q['question'])
    rows.append({
        'id':          q['id'],
        'difficulty':  q['difficulty'],
        'answerable':  q['answerable'],
        'question':    q['question'],
        'gt_value':    q.get('gt_value'),
        'gemma_value': a.get('value'),
        'values': a.get('values', []),
        'also_accept': q.get('also_accept') or [],
        'error': a.get('error'),
        'gemma_answer': a.get('answer'),
        'refused':     bool(a.get('insufficient_data')),
        'latency_s':   a['latency_s'],
    })
    print('ok')

df = pd.DataFrame(rows)
df[['id', 'gt_value', 'gemma_value', 'refused']]


## 6. Score it

A question is correct if Gemma's number is within 1% of the true value.

Evidence coverage is 0% and the unsupported-claim rate is 100% — not because we measured
them, but **by construction**: no query was executed, so nothing can be traced. Say it
that way in the report.


In [ ]:
def is_correct(row):
    if not row['answerable']:
        return None
    if row.get('error') or row['refused']:
        return False
    import math
    targets = [row['gt_value']] + row['also_accept']
    for value in row['values']:
        for target in targets:
            try:
                a, b = float(value), float(target)
                if math.isfinite(a) and math.isfinite(b) and abs(a-b) <= TOLERANCE * max(abs(a), abs(b)):
                    return True
            except (ValueError, TypeError):
                continue
    return False


df['correct'] = df.apply(is_correct, axis=1)

answerable = df[df.answerable].copy()
tricks     = df[~df.answerable].copy()
answerable['correct'] = answerable['correct'].astype(float)

summary = {
    'model':              MODEL,
    'questions':          len(df),
    'accuracy':           round(answerable.correct.mean(), 3),
    'correct_answers':    f"{int(answerable.correct.sum())}/{len(answerable)}",
    'correct_refusals':   f"{int(tricks.refused.sum())}/{len(tricks)}",
    'evidence_coverage':  0.0,
    'mean_latency_s':     round(df.latency_s.mean(), 1),
    'cost_usd':           0.0,
}
pd.Series(summary).to_frame('tier_1')


### How wrong were the numbers?

Accuracy alone hides the size of the error. This is the table to screenshot.


In [ ]:
wrong = answerable[answerable.correct == 0].copy()
wrong['off_by_%'] = ((pd.to_numeric(wrong.gemma_value, errors='coerce') - wrong.gt_value.astype(float))
                     / wrong.gt_value.astype(float) * 100).round(1)
wrong[['id', 'gt_value', 'gemma_value', 'off_by_%']]


### The 5 impossible questions

These cannot be answered from the data. Any row where `refused` is False and a value
appeared is a hallucination a business user would have acted on.


In [ ]:
tricks[['id', 'question', 'refused', 'gemma_value', 'gemma_answer']]


## 7. Chart


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))

ax[0].bar(['correct', 'wrong'],
          [answerable.correct.sum(), len(answerable) - answerable.correct.sum()],
          color=['#27ae60', '#c0392b'])
ax[0].set_title(f'{MODEL}: 25 answerable questions')
ax[0].set_ylabel('questions')

ax[1].bar(['refused\n(correct)', 'answered\n(made it up)'],
          [tricks.refused.sum(), (~tricks.refused).sum()],
          color=['#27ae60', '#c0392b'])
ax[1].set_title('5 impossible questions')

plt.tight_layout()
plt.savefig(RESULTS / 'tier1_baseline.png', dpi=150)
plt.show()


## 8. Save

`baseline_tier1.csv` is what the final comparison reads. Keep the name.


In [ ]:
df.to_csv(RESULTS / 'baseline_tier1.csv', index=False)
pd.Series(summary).to_frame('tier_1').to_csv(RESULTS / 'baseline_tier1_summary.csv')
print('saved to', RESULTS)


## 9. For the report

Write down now:

- accuracy on the 25 answerable questions
- how many of the 5 impossible ones it answered anyway
- mean latency (Tier 1 is the fastest tier — report that honestly)

And copy out **one** confident wrong answer next to the true figure. A single example
beside the real £9,809,614.01 lands harder in a presentation than any table.

**Next:** `04_tier2_agent.ipynb` — same 15 questions, but the model can run SQL.
Reuse `ask`, `is_correct` and the scoring cell so the tiers stay comparable.
